In [1]:
from datetime import datetime
import os
import pickle
import random

import pandas as pd
from pydub import AudioSegment
import seaborn as sns
import torch
import torchaudio
from torch.utils.data import (
    DataLoader, 
    Dataset, 
    random_split
)
import torch.nn as nn
from tqdm.notebook import (
    tqdm, 
    trange
)

# Prepare

In [2]:
IS_RESAMPLE = False
main_path = os.getcwd()
labeling_path = os.path.join(
    main_path,
    'project-1-at-2025-05-13-11-10-34463d27.json'
)
audio_sets_path = os.path.join(
    os.path.dirname(os.getcwd()),
    'Phonetics Project (Praat)'
    )
for _path in (main_path, labeling_path, audio_sets_path):
    display(_path)

'/home/alexeysvatov/projects/ProjectLinguist/jupyter/scripts'

'/home/alexeysvatov/projects/ProjectLinguist/jupyter/scripts/project-1-at-2025-05-13-11-10-34463d27.json'

'/home/alexeysvatov/projects/ProjectLinguist/jupyter/Phonetics Project (Praat)'

# Label

In [3]:
labels_df = pd.read_json(labeling_path)
labels_df = labels_df.loc[:, ['file_upload', 'annotations']].explode('annotations', ignore_index=True)
labels_df['file_upload'] = labels_df['file_upload'].apply(lambda _: _.partition('-')[2])
labels_df['result'] = labels_df['annotations'].apply(lambda _: _['result'] if isinstance(_, dict) else _)
labels_df = labels_df.drop('annotations', axis=1).explode('result', ignore_index=True)
labels_df['start'] = labels_df['result'].apply(lambda _: _['value']['start'] if isinstance(_, dict) else _)
labels_df['end'] = labels_df['result'].apply(lambda _: _['value']['end'] if isinstance(_, dict) else _)
labels_df['difference'] = (labels_df['end'] - labels_df['start']) * 1000
labels_df['label'] = labels_df['result'].apply(lambda _: _['value']['labels'] if isinstance(_, dict) else _)
labels_df = labels_df.drop('result', axis=1).explode('label', ignore_index=True)
labels_df = labels_df.loc[(labels_df.label != 'ignore') & (labels_df.label != 'noise') & (labels_df.end - labels_df.start >= 0.1), :]
labels_df['colab_file_path'] = audio_sets_path + '/' + labels_df['file_upload'].str.partition('_')[0] + '_' + labels_df['file_upload'].str.partition('_')[2].str.replace('_', ' ')
labels_df['label'] = pd.Categorical(labels_df['label'])
labels_df['label_id'] = labels_df['label'].cat.codes
labels_df.head()

,file_upload,start,end,difference,label,colab_file_path,label_id
2,1_In_a_restaurant.mp3,5.900917,6.019838,118.921208,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18
3,1_In_a_restaurant.mp3,6.230580,6.454639,224.059099,stressed_syllable,/home/alexeysvatov/projects/ProjectLinguist/ju...,18
4,1_In_a_restaurant.mp3,6.019880,6.230599,210.718572,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15
5,1_In_a_restaurant.mp3,6.454629,6.617893,163.263443,plain,/home/alexeysvatov/projects/ProjectLinguist/ju...,15
6,1_In_a_restaurant.mp3,6.617900,6.870468,252.567449,low_rise,/home/alexeysvatov/projects/ProjectLinguist/ju...,10


In [4]:
# makes labeled slices
labels_df['colab_samples_path'] = None
colab_file_path = None
colab_samples_path = os.path.join(main_path, 'labeled_samples')
for idx in tqdm(labels_df.index):
    audio_slice_file_path = os.path.join(colab_samples_path,
                                         f"{idx}_{labels_df.loc[idx, 'colab_file_path'].split('/')[-1]}")
    labels_df.loc[idx, 'colab_samples_path'] = audio_slice_file_path
    if IS_RESAMPLE:
        if colab_file_path != labels_df.loc[idx, 'colab_file_path']:
            colab_file_path = labels_df.loc[idx, 'colab_file_path']
        audio = AudioSegment.from_mp3(labels_df.loc[idx, 'colab_file_path'])
        start_pos = labels_df.loc[idx, 'start'] * 1000
        end_pos = labels_df.loc[idx, 'end'] * 1000
        audio_slice = audio[start_pos:end_pos]
        audio_slice.export(audio_slice_file_path, format='mp3')
labels_df = labels_df.reset_index(names='old_index')

  0%|          | 0/882 [00:00<?, ?it/s]

# PyTorch

In [5]:
OUT_FEATURES = labels_df.label_id.nunique()
MAX_DURATION = 4000 # int(labels_df['difference'].max())
NORMALISE_TYPE = 'base'

In [6]:
class AudioUtil:
    # ----------------------------
    # Load an audio file. Return the signal as a tensor and the sample rate
    # ----------------------------
    @staticmethod
    def open(audio_file):
        sig, sr = torchaudio.load(audio_file)
        return sig, sr

    # ----------------------------
    # Convert the given audio to the desired number of channels
    # ----------------------------
    @staticmethod
    def rechannel(aud, new_channel):
        sig, sr = aud

        if sig.shape[0] == new_channel:
            # Nothing to do
            return aud

        if new_channel == 1:
            # Convert from stereo to mono by selecting only the first channel
            resig = sig[:1, :]
        else:
            # Convert from mono to stereo by duplicating the first channel
            resig = torch.cat([sig, sig])

        return resig, sr

    # ----------------------------
    # Since Resample applies to a single channel, we resample one channel at a time
    # ----------------------------
    @staticmethod
    def resample(aud, newsr):
        sig, sr = aud

        if sr == newsr:
            # Nothing to do
            return aud

        num_channels = sig.shape[0]
        # Resample first channel
        resig = torchaudio.transforms.Resample(sr, newsr)(sig[:1, :])
        if num_channels > 1:
            # Resample the second channel and merge both channels
            retwo = torchaudio.transforms.Resample(sr, newsr)(sig[1:, :])
            resig = torch.cat([resig, retwo])

        return resig, newsr

    # ----------------------------
    # Pad (or truncate) the signal to a fixed length 'max_ms' in milliseconds
    # ----------------------------
    @staticmethod
    def pad_trunc(aud, max_ms):
        sig, sr = aud
        num_rows, sig_len = sig.shape
        max_len = sr // 1000 * max_ms

        if sig_len > max_len:
            # Truncate the signal to the given length
            sig = sig[:, :max_len]

        elif sig_len < max_len:
            # Length of padding to add at the beginning and end of the signal
            pad_begin_len = random.randint(0, max_len - sig_len)
            pad_end_len = max_len - sig_len - pad_begin_len

            # Pad with 0s
            pad_begin = torch.zeros((num_rows, pad_begin_len))
            pad_end = torch.zeros((num_rows, pad_end_len))

            sig = torch.cat((pad_begin, sig, pad_end), 1)

        return sig, sr

    # ----------------------------
    # Shifts the signal to the left or right by some percent. Values at the end
    # are 'wrapped around' to the start of the transformed signal.
    # ----------------------------
    @staticmethod
    def time_shift(aud, shift_limit):
        sig, sr = aud
        _, sig_len = sig.shape
        shift_amt = int(random.random() * shift_limit * sig_len)
        return sig.roll(shift_amt), sr

    # ----------------------------
    # Generate a Spectrogram
    # ----------------------------
    @staticmethod
    def spectro_gram(aud, n_mels=64, n_fft=1024, hop_len=None):
        sig, sr = aud
        top_db = 80

        # spec has shape [channel, n_mels, time], where channel is mono, stereo etc
        spec = torchaudio.transforms.MelSpectrogram(sr, n_fft=n_fft, hop_length=hop_len, n_mels=n_mels)(sig)

        # Convert to decibels
        spec = torchaudio.transforms.AmplitudeToDB(top_db=top_db)(spec)
        return spec

    # ----------------------------
    # Augment the Spectrogram by masking out some sections of it in both the frequency
    # dimension (ie. horizontal bars) and the time dimension (vertical bars) to prevent
    # overfitting and to help the model generalise better. The masked sections are
    # replaced with the mean value.
    # ----------------------------
    @staticmethod
    def spectro_augment(spec, max_mask_pct=0.1, n_freq_masks=1, n_time_masks=1):
        _, n_mels, n_steps = spec.shape
        mask_value = spec.mean()
        aug_spec = spec

        freq_mask_param = max_mask_pct * n_mels
        for _ in range(n_freq_masks):
            aug_spec = torchaudio.transforms.FrequencyMasking(freq_mask_param)(aug_spec, mask_value)

        time_mask_param = max_mask_pct * n_steps
        for _ in range(n_time_masks):
            aug_spec = torchaudio.transforms.TimeMasking(time_mask_param)(aug_spec, mask_value)

        return aug_spec

In [7]:
# ----------------------------
# Sound Dataset
# ----------------------------
class SoundDS(Dataset):
    def __init__(self, df):
        self.df = df
        self.duration = MAX_DURATION  # default 4000
        self.sr = 44100
        self.channel = 2
        self.shift_pct = 0.4

    # ----------------------------
    # Number of items in dataset
    # ----------------------------
    def __len__(self):
        return len(self.df)

    # ----------------------------
    # Get i'th item in dataset
    # ----------------------------
    def __getitem__(self, idx):
        # Absolute file path of the audio file - concatenate the audio directory with
        # the relative path
        audio_file_path = self.df.loc[idx, 'colab_samples_path']
        # Get the Class ID
        class_id = self.df.loc[idx, 'label_id']

        aud = AudioUtil.open(audio_file_path)
        # Some sounds have a higher sample rate, or fewer channels compared to the
        # majority. So make all sounds have the same number of channels and same
        # sample rate. Unless the sample rate is the same, the pad_trunc will still
        # result in arrays of different lengths, even though the sound duration is
        # the same.

        reaud = AudioUtil.resample(aud, self.sr)
        rechan = AudioUtil.rechannel(reaud, self.channel)

        dur_aud = AudioUtil.pad_trunc(rechan, self.duration)  
        shift_aud = AudioUtil.time_shift(dur_aud, self.shift_pct)
        sgram = AudioUtil.spectro_gram(shift_aud, n_mels=64, n_fft=1024, hop_len=None)
        # aug_sgram = AudioUtil.spectro_augment(sgram, max_mask_pct=0.1, n_freq_masks=1, n_time_masks=1)

        return sgram, class_id # set needed variable

In [8]:
myds = SoundDS(labels_df)

# Random split of 80:20 between training and validation
num_items = len(myds)
num_train = round(num_items * 0.8)
num_val = num_items - num_train
train_ds, val_ds = random_split(myds, [num_train, num_val])

# Create training and validation data loaders
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)

In [9]:
class AudioClassifier(nn.Module):
    # ----------------------------
    # Build the model architecture
    # ----------------------------
    def __init__(self):
        super().__init__()
        conv_layers = []

        # First Convolution Block with Relu and Batch Norm. Use Kaiming Initialization
        self.conv1 = nn.Conv2d(2, 8, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2)) # for 2 channels
        self.relu1 = nn.ReLU()
        # self.sig1 = nn.Sigmoid()
        self.bn1 = nn.BatchNorm2d(8)
        nn.init.kaiming_normal_(self.conv1.weight, a=0.1)
        self.conv1.bias.data.zero_()
        conv_layers += [
            self.conv1, 
            self.relu1, 
            # self.sig1,
            self.bn1,
        ]

        # Second Convolution Block
        self.conv2 = nn.Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)) # for 2 channels
        self.relu2 = nn.ReLU()
        # self.sig2 = nn.Sigmoid()
        self.bn2 = nn.BatchNorm2d(16)
        nn.init.kaiming_normal_(self.conv2.weight, a=0.1)
        self.conv2.bias.data.zero_()
        conv_layers += [
            self.conv2, 
            self.relu2, 
            # self.sig2, 
            self.bn2,
        ]

        # Third Convolution Block
        self.conv3 = nn.Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)) # for 2 channels
        self.relu3 = nn.ReLU()
        # self.sig3 = nn.Sigmoid()
        self.bn3 = nn.BatchNorm2d(32)
        nn.init.kaiming_normal_(self.conv3.weight, a=0.1)
        self.conv3.bias.data.zero_()
        conv_layers += [
            self.conv3, 
            self.relu3, 
            # self.sig3,
            self.bn3,
        ]

        # Fourth Convolution Block
        self.conv4 = nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)) # for 2 channels
        # self.relu4 = nn.ReLU()
        self.sig4 = nn.Sigmoid()
        self.bn4 = nn.BatchNorm2d(64)
        nn.init.kaiming_normal_(self.conv4.weight, a=0.1)
        self.conv4.bias.data.zero_()
        conv_layers += [
            self.conv4, 
            # self.relu4, 
            self.sig4,
            self.bn4
        ]

        # # Fifth Convolution Block
        # self.conv5 = nn.Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1)) # for 2 channels
        # # self.relu5 = nn.ReLU()
        # self.sig5 = nn.Sigmoid()
        # self.bn5 = nn.BatchNorm2d(128)
        # nn.init.kaiming_normal_(self.conv5.weight, a=0.1)
        # self.conv5.bias.data.zero_()
        # conv_layers += [
        #     self.conv5, 
        #     # self.relu5, 
        #     self.sig5,
        #     self.bn5
        # ]

        # Linear Classifier
        self.ap = nn.AdaptiveAvgPool2d(output_size=1)
        self.lin = nn.Linear(in_features=64, out_features=OUT_FEATURES)

        # Wrap the Convolutional Blocks
        self.conv = nn.Sequential(*conv_layers)

    # ----------------------------
    # Forward pass computations
    # ----------------------------
    def forward(self, x):
        # Run the convolutional blocks
        x = self.conv(x)

        # Adaptive pool and flatten for input to linear layer
        x = self.ap(x)
        x = x.view(x.shape[0], -1)

        # Linear layer
        x = self.lin(x)

        # Final output
        return x

In [10]:
# Create the model and put it on the GPU if available
myModel = AudioClassifier()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
myModel = myModel.to(device)
# Check that it is on Cuda
next(myModel.parameters()).device

device(type='cpu')

In [11]:
def normalise(_inputs, _type: str | None):
    if _type == 'base':
        return (_inputs - _inputs.mean()) / _inputs.std()
    else:
        return _inputs

In [12]:
# ----------------------------
# Training Loop
# ----------------------------
def training(model, train_dl, num_epochs, normalise_type: str | None = None):
    model.train()
    
    # Loss Function, Optimizer and Scheduler
    criterion = nn.CrossEntropyLoss()
    # optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1,
                                                    steps_per_epoch=int(len(train_dl)),
                                                    epochs=num_epochs,
                                                    anneal_strategy='linear')
    stats = {
        'epoch': [],
        'avg_loss': [],
        'acc': []
    }
    # Repeat for each epoch
    for epoch in trange(num_epochs):
        running_loss = 0.0
        correct_prediction = 0
        total_prediction = 0

        # Repeat for each batch in the training set
        for data in train_dl:
            # Get the input features and target labels, and put them on the GPU
            inputs, labels = data[0].to(device), data[1].to(device).long()

            # Normalize the inputs
            inputs = normalise(inputs, _type=normalise_type)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            # Keep stats for Loss and Accuracy
            running_loss += loss.item()

            # Get the predicted class with the highest score
            # _, prediction = torch.max(outputs, 1)
            prediction = torch.softmax(outputs, dim=1).argmax(dim=1)
            
            # Count of predictions that matched the target label
            correct_prediction += (prediction == labels).sum().item()
            total_prediction += prediction.shape[0]

        # collect stats
        num_batches = len(train_dl)
        avg_loss = running_loss / num_batches
        acc = correct_prediction / total_prediction
        stats['epoch'].append(epoch)
        stats['avg_loss'].append(avg_loss)
        stats['acc'].append(acc)

    display('Finished Training')
    return stats

In [13]:
# ----------------------------
# Inference
# ----------------------------
def inference(model, val_dl, normalise_type: str | None = None):
    model.eval()
    correct_prediction = 0
    total_prediction = 0

    # Disable gradient updates
    with torch.no_grad():
        for data in tqdm(val_dl):
            # Get the input features and target labels, and put them on the GPU
            inputs, labels = data[0].to(device), data[1].to(device)

            # Normalize the inputs
            inputs = normalise(inputs, _type=normalise_type)

            # Get predictions
            outputs = model(inputs)

            # Get the predicted class with the highest score
            # _, prediction = torch.max(outputs, 1)
            prediction = torch.softmax(outputs, dim=1).argmax(dim=1)

            # Count of predictions that matched the target label
            correct_prediction += (prediction == labels).sum().item()
            total_prediction += prediction.shape[0]

    acc = correct_prediction / total_prediction
    display(f'Accuracy: {acc:.2f}, Total items: {total_prediction}')

In [ ]:
num_epochs = 10
stats = training(myModel, train_dl, num_epochs, normalise_type=NORMALISE_TYPE)

  0%|          | 0/10 [00:00<?, ?it/s]

# Checks

In [ ]:
stats_df = pd.DataFrame(stats)
stats_df.head()

In [ ]:
sns.lineplot(
    data=stats_df,
    x='epoch',
    y='acc',
)

In [ ]:
sns.lineplot(
    data=stats_df,
    x='epoch',
    y='avg_loss',
)

In [ ]:
# Run inference on trained model with the validation set
inference(myModel, val_dl, normalise_type=NORMALISE_TYPE)

# Save model and stats

In [ ]:
stats_dir_name = 'dl_stats'
dir_name, _, _ = datetime.now().isoformat().replace('T', '_').replace(':', '-').rpartition('.')
cur_stats_dir_name = os.path.join(stats_dir_name, dir_name)
os.makedirs(cur_stats_dir_name)
with open(os.path.join(cur_stats_dir_name, 'AudioClassifier.pickle'), 'wb') as f:
    pickle.dump(myModel, f)
stats_df.to_parquet(os.path.join(cur_stats_dir_name, 'statistics.parquet'))